# Fisher Statistics for Paleomagnetic Data

Statistical analysis is fundamental to any paleomagnetic investigation. We require methods for determining a mean direction from a set of observations, quantifying the uncertainty of that mean, and for assessing the significance of field tests of paleomagnetic stability.

Most of the statistical methods used in paleomagnetism have direct analogies to "planar" statistics. The key difference is that paleomagnetic data are *directional* — they are unit vectors on a sphere rather than scalar values on a number line. R.A. Fisher developed a probability density function applicable to directional data sets that is the spherical analogue of the normal distribution (Fisher, 1953). This **Fisher distribution** (also called the von Mises-Fisher distribution) underpins the statistical framework used throughout paleomagnetism.

In this notebook, we will:
1. Explore the Fisher probability density function and its parameters
2. Generate synthetic Fisher-distributed data using PmagPy
3. Calculate Fisher statistics ($k$, $\alpha_{95}$, CSD) and develop intuition for what they mean
4. Investigate how sample size affects our statistical estimates
5. Apply significance tests for comparing directions
6. Test whether data are consistent with a Fisher distribution

In [ ]:
!pip install pmagpy
!pip install cartopy

In [ ]:
import pmagpy.pmag as pmag
import pmagpy.ipmag as ipmag
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display

%matplotlib inline

## Why do we need Fisher statistics?

Whenever we work with directional data — unit vectors on a sphere — standard Gaussian statistics do not apply. Directions arise throughout paleomagnetism: magnetization directions from demagnetization experiments, virtual geomagnetic poles (VGPs), paleomagnetic poles, and structural orientations. The Fisher distribution provides a statistical framework for calculating means, quantifying scatter, and placing confidence limits on such directional data.

To illustrate the importance of this framework, consider paleomagnetic poles. We measure the magnetization direction of many individual lava flows from a rock unit, each yielding a VGP — an estimate of the geomagnetic pole position at the time of magnetization. These VGPs are scattered due to paleosecular variation of the geomagnetic field as well as measurement uncertainty. Fisher statistics let us calculate a mean pole with a confidence circle that succinctly quantifies what we know about the pole position.

Here are real VGPs from ca. 1.1 billion-year-old volcanic rocks at Mamainse Point, Ontario (Swanson-Hysell et al., 2014). Two groups of lava flows — early reversed-polarity and late normal-polarity — record different pole positions as the continent moved:

In [ ]:
data_url = 'https://raw.githubusercontent.com/Swanson-Hysell/Fisher_Practical_Stats/refs/heads/master/data/Swanson-Hysell2014a/sites.txt'
MP_sites = pd.read_csv(data_url,
                       sep='\t', header=1)

# Two subsets: early reversed polarity and late normal polarity
early_R = MP_sites[MP_sites.height < 600]
late_N = MP_sites[MP_sites.height > 2400]

# Plot the individual VGPs on an orthographic map
m = ipmag.make_orthographic_map(central_latitude=35, central_longitude=200)
ipmag.plot_vgp(m, early_R.vgp_lon.tolist(), early_R.vgp_lat.tolist(),
               color='#930E26', label=f'Early reversed (N={len(early_R)})')
ipmag.plot_vgp(m, late_N.vgp_lon.tolist(), late_N.vgp_lat.tolist(),
               color='#3B7185', label=f'Late normal (N={len(late_N)})')
plt.legend(loc='lower center')
plt.title('VGPs from Mamainse Point volcanics (~1.1 Ga)', fontsize=13)
plt.show()

Each dot is a VGP from a single lava flow. There is scatter within each group as expected through geomagnetic secular variation. Fisher statistics allow us to calculate a mean pole and a 95% confidence circle ($A_{95}$) for each group of VGPs, summarizing our knowledge of where the pole was at each time:

In [ ]:
# Calculate Fisher means for each group
mean_early = ipmag.fisher_mean(dec=early_R.vgp_lon.tolist(),
                                inc=early_R.vgp_lat.tolist())
mean_late = ipmag.fisher_mean(dec=late_N.vgp_lon.tolist(),
                               inc=late_N.vgp_lat.tolist())

# Plot VGPs with Fisher mean poles and A95 confidence circles
m = ipmag.make_orthographic_map(central_latitude=35, central_longitude=200)

ipmag.plot_vgp(m, early_R.vgp_lon.tolist(), early_R.vgp_lat.tolist(),
               color='#930E26', label=f'Early reversed (N={len(early_R)})')
ipmag.plot_pole(m, mean_early['dec'], mean_early['inc'], mean_early['alpha95'],
                marker='s', color='#930E26', edgecolor='k', markersize=30)

ipmag.plot_vgp(m, late_N.vgp_lon.tolist(), late_N.vgp_lat.tolist(),
               color='#3B7185', label=f'Late normal (N={len(late_N)})')
ipmag.plot_pole(m, mean_late['dec'], mean_late['inc'], mean_late['alpha95'],
                marker='s', color='#3B7185', edgecolor='k', markersize=30)

plt.legend(loc='lower center')
plt.title('Fisher mean poles with $A_{95}$ confidence circles', fontsize=13)
plt.show()

print(f"Early reversed: PLon={mean_early['dec']:.1f}°, PLat={mean_early['inc']:.1f}°, "
      f"k={mean_early['k']:.1f}, A95={mean_early['alpha95']:.1f}°, N={mean_early['n']}")
print(f"Late normal:    PLon={mean_late['dec']:.1f}°, PLat={mean_late['inc']:.1f}°, "
      f"k={mean_late['k']:.1f}, A95={mean_late['alpha95']:.1f}°, N={mean_late['n']}")

The $A_{95}$ confidence circles (squares with surrounding circles) succinctly summarize our knowledge of the pole position at each time. These circles are small enough to resolve the motion of the pole between the two time intervals — this is what makes it possible to reconstruct the movement of continents through Earth history.

But how are these mean poles and confidence circles calculated? What assumptions go into them? How many samples do we need? In the rest of this notebook, we develop the statistical framework — the **Fisher distribution** — that underlies these calculations.

## The Fisher Probability Density Function

In Fisher statistics, each direction is given unit weight and is represented by a point on a sphere of unit radius. The Fisher distribution function $P_{dA}(\alpha)$ gives the probability per unit angular area of finding a direction within an angular area $dA$ centered at an angle $\alpha$ from the true mean:

$$P_{dA}(\alpha) = \frac{\kappa}{4\pi \sinh \kappa} \exp (\kappa \cos \alpha)$$

where $\alpha$ is the angle between the unit vector and the true mean direction and $\kappa$ is the **precision parameter**. As $\kappa \to \infty$, dispersion goes to zero (all directions cluster at the mean); $\kappa = 0$ corresponds to a uniform distribution over the sphere.

Because of the geometry of the sphere, the probability of finding a direction in a *band* of width $d\alpha$ at angle $\alpha$ from the mean picks up a $\sin\alpha$ factor from the angular area element:

$$P_{d\alpha}(\alpha) = \frac{\kappa}{2 \sinh \kappa} \exp (\kappa \cos \alpha) \sin \alpha$$

This $\sin\alpha$ term means that the most probable angle is not zero — there is more area on the sphere at moderate angles than right at the pole. The interplay between the exponential decay and the $\sin\alpha$ factor is key to understanding the Fisher distribution.

In [ ]:
def fisher_pdf_area(alpha_deg, kappa):
    """P_dA(alpha) - probability per unit angular area."""
    alpha = np.deg2rad(alpha_deg)
    return (kappa / (4 * np.pi * np.sinh(kappa))) * np.exp(kappa * np.cos(alpha))

def fisher_pdf_band(alpha_deg, kappa):
    """P_dalpha(alpha) - probability in a band of width dalpha."""
    alpha = np.deg2rad(alpha_deg)
    return (kappa / (2 * np.sinh(kappa))) * np.exp(kappa * np.cos(alpha)) * np.sin(alpha)

def plot_fisher_pdf(kappa_values=[5, 10, 50, 100]):
    """Plot the Fisher PDF for given kappa values."""
    alpha = np.linspace(0, 90, 500)
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    for kappa, color in zip(kappa_values, colors):
        ax1.plot(alpha, fisher_pdf_area(alpha, kappa), color=color,
                 label=f'$\\kappa$ = {kappa}', linewidth=2)
        ax2.plot(alpha, fisher_pdf_band(alpha, kappa), color=color,
                 label=f'$\\kappa$ = {kappa}', linewidth=2)

    ax1.set_xlabel('Angle from true mean ($\\alpha$)', fontsize=12)
    ax1.set_ylabel('$P_{dA}(\\alpha)$', fontsize=12)
    ax1.set_title('Probability per unit angular area', fontsize=13)
    ax1.legend(fontsize=11)

    ax2.set_xlabel('Angle from true mean ($\\alpha$)', fontsize=12)
    ax2.set_ylabel('$P_{d\\alpha}(\\alpha)$', fontsize=12)
    ax2.set_title('Probability per angular band', fontsize=13)
    ax2.legend(fontsize=11)

    plt.tight_layout()
    plt.show()

plot_fisher_pdf()

Notice the difference between the two panels:
- **Left panel**: The probability per unit angular area $P_{dA}$ is highest right at the mean ($\alpha = 0$) and decays exponentially. This is the "intrinsic" concentration of the distribution.
- **Right panel**: The probability per angular band $P_{d\alpha}$ goes to zero at $\alpha = 0$ because the band area ($\propto \sin\alpha$) vanishes at the pole. For lower $\kappa$ values, the peak shifts to larger angles — most directions are found at moderate angles from the mean, not right at the mean.

Use the widget below to explore how the Fisher PDF changes with $\kappa$:

In [ ]:
def interactive_fisher_pdf(kappa=20):
    """Interactively explore the Fisher PDF for a single kappa value."""
    alpha = np.linspace(0.1, 90, 500)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    ax1.plot(alpha, fisher_pdf_area(alpha, kappa), color='steelblue', linewidth=2)
    ax1.fill_between(alpha, fisher_pdf_area(alpha, kappa), alpha=0.2, color='steelblue')
    ax1.set_xlabel('Angle from true mean ($\\alpha$)', fontsize=12)
    ax1.set_ylabel('$P_{dA}(\\alpha)$', fontsize=12)
    ax1.set_title(f'Probability per unit area ($\\kappa$ = {kappa})', fontsize=13)
    ax1.set_xlim(0, 90)

    ax2.plot(alpha, fisher_pdf_band(alpha, kappa), color='#d62728', linewidth=2)
    ax2.fill_between(alpha, fisher_pdf_band(alpha, kappa), alpha=0.2, color='#d62728')
    ax2.set_xlabel('Angle from true mean ($\\alpha$)', fontsize=12)
    ax2.set_ylabel('$P_{d\\alpha}(\\alpha)$', fontsize=12)
    ax2.set_title(f'Probability per angular band ($\\kappa$ = {kappa})', fontsize=13)
    ax2.set_xlim(0, 90)

    # Mark the peak of P_dalpha
    peak_idx = np.argmax(fisher_pdf_band(alpha, kappa))
    peak_angle = alpha[peak_idx]
    ax2.axvline(peak_angle, color='gray', linestyle='--', alpha=0.7)
    ax2.annotate(f'peak at {peak_angle:.1f}°', xy=(peak_angle, fisher_pdf_band(peak_angle, kappa)),
                 xytext=(peak_angle + 10, fisher_pdf_band(peak_angle, kappa) * 0.8),
                 fontsize=11, arrowprops=dict(arrowstyle='->', color='gray'))

    plt.tight_layout()
    plt.show()

kappa_slider = widgets.IntSlider(value=20, min=1, max=500, step=1,
                                  description='κ:', style={'description_width': 'initial'},
                                  continuous_update=True)
widgets.interactive(interactive_fisher_pdf, kappa=kappa_slider)

## Generating Fisher-Distributed Data

PmagPy provides the function `ipmag.fishrot()` to draw random directions from a Fisher distribution with a specified true mean direction ($D$, $I$) and precision parameter $\kappa$. Each call returns a different random sample — just as taking different sets of paleomagnetic samples from the same rock unit would yield different sets of measured directions.

Let's generate three sets of directions from Fisher distributions with the same vertical true mean but different $\kappa$ values to see how the concentration parameter controls the scatter:

In [ ]:
n = 30
true_dec, true_inc = 0, 90  # vertical true mean direction

def plot_fisher_sample(kappa, n=30, true_dec=0, true_inc=90):
    """Draw N directions from a Fisher distribution and plot on a stereonet."""
    directions = ipmag.fishrot(k=kappa, n=n, dec=true_dec, inc=true_inc)
    mean = ipmag.fisher_mean(di_block=directions)

    plt.figure(figsize=(5, 5))
    ipmag.plot_net(fignum=0)
    ipmag.plot_di(di_block=directions, color='steelblue', markersize=30, edge='k')
    ipmag.plot_di_mean(mean['dec'], mean['inc'], mean['alpha95'],
                       color='red', marker='s', markersize=40)
    plt.title(
        f"$\\kappa$ = {kappa}\n"
        f"$\\bar{{D}}$ = {mean['dec']:.1f}°, $\\bar{{I}}$ = {mean['inc']:.1f}°\n"
        f"k = {mean['k']:.1f}, $\\alpha_{{95}}$ = {mean['alpha95']:.1f}°",
        fontsize=12
    )
    plt.show()

In [ ]:
plot_fisher_sample(kappa=5, n=30, true_dec=0, true_inc=90)

In [ ]:
plot_fisher_sample(kappa=10, n=30, true_dec=0, true_inc=90)

In [ ]:
plot_fisher_sample(kappa=50, n=30, true_dec=0, true_inc=90)

**Try it yourself:** In the cell below, experiment with calling `plot_fisher_sample()` with different parameters. The function signature is `plot_fisher_sample(kappa, n=30, true_dec=0, true_inc=90)`. Try changing:
- **$\kappa$**: Start large (e.g., 200) and work down to small values (e.g., 2). At what $\kappa$ does the $\alpha_{95}$ circle become very large?
- **n**: How does increasing or decreasing the number of directions affect $k$ and $\alpha_{95}$?
- **true_dec, true_inc**: Change the true mean direction away from vertical. Notice that the estimated mean doesn't end up being exactly the true mean.

In [ ]:
plot_fisher_sample(kappa=10, n=30, true_dec=45, true_inc=45)

## Estimating Fisher Statistics

Given a set of $N$ unit vectors (directions), the Fisher statistics are estimated as follows:

### The resultant vector $R$

The mean direction is calculated by converting individual directions ($D_i, I_i$) to Cartesian coordinates, summing them as vectors, and normalizing. The length of the vector sum — the **resultant** $R$ — is:

$$R^2 = \left(\sum_i x_{1i}\right)^2 + \left(\sum_i x_{2i}\right)^2 + \left(\sum_i x_{3i}\right)^2$$

$R$ is always $\leq N$ and approaches $N$ only when the vectors are tightly clustered.

![Vector addition of unit vectors to yield resultant vector R](https://raw.githubusercontent.com/PmagPy/Essentials-JupyterBook/main/book/figures/chapter11/vecsum.png)

### Precision parameter $k$

The best estimate of $\kappa$ from a finite sample is:

$$k = \frac{N-1}{N-R}$$

$k$ is *not* fundamentally dependent on $N$ — it estimates an intrinsic property of the population from which the data were drawn.

### Circle of 95% confidence $\alpha_{95}$

$$\alpha_{95} = \cos^{-1}\left[1 - \frac{N-R}{R}\left[\left(\frac{1}{0.05}\right)^{1/(N-1)}-1\right]\right]$$

$\alpha_{95}$ *does* depend on $N$: the more directions you measure, the more precisely you estimate the true mean, and $\alpha_{95}$ decreases approximately as $1/\sqrt{N}$. A classic approximation valid for $k > 25$ is:

$$\alpha_{95} \approx \frac{140}{\sqrt{kN}}$$

### Circular Standard Deviation (CSD)

The CSD estimates the angular standard deviation of the population — the circle containing ~68% of the data:

$$\text{CSD} \approx \frac{81}{\sqrt{k}}$$

Like $k$, the CSD estimates an intrinsic property of the population and is not fundamentally dependent on $N$.

### How statistics depend on $N$

The key distinction is between statistics that estimate *population parameters* (like $k$ and CSD, which characterize the intrinsic scatter) and statistics that estimate *precision of the mean* (like $\alpha_{95}$, which improves with more data). The following plot illustrates this by progressively adding directions from a single Fisher-distributed sample.

**Run the cell below multiple times** — each execution draws a new random sample. Try changing $\kappa$ as well. Notice how CSD fluctuates around the true value while $\alpha_{95}$ consistently decreases with $N$:

In [ ]:
def plot_stats_vs_n(kappa=30, n_total=50):
    """Draw a Fisher-distributed sample and show how CSD and alpha95
    evolve as directions are progressively added."""
    S_true = 81 / np.sqrt(kappa)
    directions = ipmag.fishrot(k=kappa, n=n_total, dec=0, inc=90)

    ns = list(range(4, n_total + 1))
    csds = []
    a95s = []

    for n in ns:
        subset = directions[:n]
        mean = ipmag.fisher_mean(di_block=subset)
        csds.append(mean['csd'])
        a95s.append(mean['alpha95'])

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    ax1.plot(ns, csds, 'o-', color='steelblue', label='CSD', markersize=4)
    ax1.axhline(y=S_true, color='gray', linestyle='--',
                label=f'True S = {S_true:.1f}°', linewidth=2)
    ax1.set_xlabel('N (number of directions)', fontsize=12)
    ax1.set_ylabel('Angle (degrees)', fontsize=12)
    ax1.set_title('Dispersion estimate (CSD) vs. N', fontsize=13)
    ax1.legend(fontsize=11)

    ax2.plot(ns, a95s, 's-', color='#d62728', label='$\\alpha_{95}$', markersize=4)
    ax2.set_xlabel('N (number of directions)', fontsize=12)
    ax2.set_ylabel('$\\alpha_{95}$ (degrees)', fontsize=12)
    ax2.set_title('Confidence limit vs. N', fontsize=13)
    ax2.legend(fontsize=11)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_stats_vs_n(kappa=30, n_total=50)

Key observations:
- **CSD** fluctuates around the true angular standard deviation $S$ but is not fundamentally dependent on $N$. It can deviate substantially from $S$ for small $N$ (< ~15), only converging to the correct value with sufficient data.
- **$\alpha_{95}$** decreases approximately as $1/\sqrt{N}$, showing a dramatic decrease for $4 < N < 10$ and more gradual improvement for larger $N$.

This has practical implications: adding a few more samples to a small data set dramatically improves your confidence limit, but there are diminishing returns beyond ~20-30 directions.

## Sampling Variability

An essential concept is the distinction between the statistics we *calculate* from a data set and the unknown *parameters* of the sampled population. Every time we sample the same formation, we get a slightly different set of directions and slightly different statistics. The following widget draws many independent samples from the same Fisher distribution and shows the resulting distributions of $k$, $\alpha_{95}$, and the angular deviation of the mean from the true direction:

In [ ]:
def repeated_sampling(kappa=20, N=20, n_experiments=100):
    """Draw many independent samples from the same Fisher distribution
    and examine the distribution of estimated statistics."""
    true_dec, true_inc = 0, 90
    ks = []
    a95s = []
    angular_devs = []

    for _ in range(n_experiments):
        dirs = ipmag.fishrot(k=kappa, n=N, dec=true_dec, inc=true_inc)
        mean = ipmag.fisher_mean(di_block=dirs)
        ks.append(mean['k'])
        a95s.append(mean['alpha95'])
        # angular deviation of estimated mean from true direction
        dev = np.degrees(np.arccos(
            np.sin(np.radians(mean['inc'])) * np.sin(np.radians(true_inc)) +
            np.cos(np.radians(mean['inc'])) * np.cos(np.radians(true_inc)) *
            np.cos(np.radians(mean['dec'] - true_dec))
        ))
        angular_devs.append(dev)

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    axes[0].hist(ks, bins=20, color='steelblue', edgecolor='white')
    axes[0].axvline(kappa, color='red', linestyle='--', linewidth=2,
                    label=f'True $\\kappa$ = {kappa}')
    axes[0].set_xlabel('Estimated k', fontsize=12)
    axes[0].set_ylabel('Count', fontsize=12)
    axes[0].set_title('Distribution of k', fontsize=13)
    axes[0].legend()

    axes[1].hist(a95s, bins=20, color='#d62728', edgecolor='white')
    axes[1].set_xlabel('$\\alpha_{95}$ (degrees)', fontsize=12)
    axes[1].set_ylabel('Count', fontsize=12)
    axes[1].set_title('Distribution of $\\alpha_{95}$', fontsize=13)

    n_outside = sum(1 for d, a in zip(angular_devs, a95s) if d > a)
    axes[2].hist(angular_devs, bins=20, color='#2ca02c', edgecolor='white')
    axes[2].set_xlabel('Angle from true mean (degrees)', fontsize=12)
    axes[2].set_ylabel('Count', fontsize=12)
    axes[2].set_title(
        f'Deviation of mean from truth\n'
        f'{n_outside}/{n_experiments} outside $\\alpha_{{95}}$'
        f' ({100*n_outside/n_experiments:.0f}%)',
        fontsize=12
    )

    plt.tight_layout()
    plt.show()

style = {'description_width': 'initial'}
kappa_w = widgets.IntSlider(value=20, min=5, max=200, step=5,
                             description='κ:', style=style, continuous_update=False)
n_w = widgets.IntSlider(value=20, min=5, max=100, step=5,
                         description='N:', style=style, continuous_update=False)
nexp_w = widgets.IntSlider(value=100, min=20, max=500, step=20,
                            description='Experiments:', style=style, continuous_update=False)

widgets.interactive(repeated_sampling, kappa=kappa_w, N=n_w, n_experiments=nexp_w)

The right panel should show that approximately 5% of experiments have their estimated mean outside the $\alpha_{95}$ circle — this is exactly what "95% confidence" means. Try increasing $N$ and observe how the distribution of $k$ tightens around the true $\kappa$ and $\alpha_{95}$ values decrease.

## Properties of the Fisher Distribution

The Fisher distribution has two distinctive properties when viewed in "data coordinates" (i.e., rotated so the mean is at the center):
1. **Declinations** (azimuths about the mean) are **uniformly distributed** from 0° to 360°
2. **Co-inclinations** (angles from the mean, $\alpha$) are **exponentially distributed**

We can verify this by drawing a large sample and plotting histograms alongside the expected distributions:

In [ ]:
def plot_fisher_properties(kappa=50, N=10000):
    """Draw a large Fisher-distributed sample and compare declination
    and co-inclination distributions to theoretical expectations."""
    # Generate directions with vertical mean (dec=0, inc=90)
    # so declination = azimuth and co-inclination = 90 - inc
    directions = ipmag.fishrot(k=kappa, n=N, dec=0, inc=90)
    lon, lat, _ = ipmag.unpack_di_block(directions)

    co_inc = 90 - np.array(lat)  # angle from the pole
    lon_array = np.array(lon)

    lon_bins = 36
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    # Declination histogram
    ax1.hist(lon_array, bins=lon_bins, color='steelblue', edgecolor='white',
             density=True, alpha=0.7)
    ax1.axhline(1/360, color='#ff7f0e', linewidth=2,
                label='Expected uniform', linestyle='--')
    ax1.set_xlabel('Declination (°)', fontsize=12)
    ax1.set_ylabel('Probability density', fontsize=12)
    ax1.set_title('Declinations: uniform distribution', fontsize=13)
    ax1.legend(fontsize=11)

    # Co-inclination histogram with expected Fisher curve
    ax2.hist(co_inc, bins=30, color='#2ca02c', edgecolor='white',
             density=True, alpha=0.7)
    alpha_range = np.linspace(0, max(co_inc), 200)
    expected = fisher_pdf_band(alpha_range, kappa)
    # Normalize to match probability density (integrate to 1 over degrees)
    expected_norm = expected / np.trapezoid(expected, alpha_range)
    ax2.plot(alpha_range, expected_norm, color='#ff7f0e', linewidth=2,
             label=f'Expected ($\\kappa$={kappa})', linestyle='--')
    ax2.set_xlabel('Co-inclination α (°)', fontsize=12)
    ax2.set_ylabel('Probability density', fontsize=12)
    ax2.set_title('Co-inclinations: exponential distribution', fontsize=13)
    ax2.legend(fontsize=11)

    plt.tight_layout()
    plt.show()

style = {'description_width': 'initial'}
kappa_w = widgets.IntSlider(value=50, min=5, max=200, step=5,
                             description='κ:', style=style, continuous_update=True)

widgets.interactive(plot_fisher_properties, kappa=kappa_w,
                    N=widgets.fixed(10000))

## Significance Tests

The Fisher distribution allows us to ask several important questions about paleomagnetic data sets:
1. Is a given set of directions **random**? (e.g., conglomerate test)
2. Is one data set **better grouped** than another? (e.g., fold test)
3. Is the mean direction **different from a known direction**? (e.g., comparison to GAD field)
4. Are **two data sets different** antiparallel from each other? (e.g., reversals test)

Two fundamental principles apply to parametric significance tests:
1. Tests compare observations against a **null hypothesis**. They do not *prove* the null hypothesis wrong — they only show that observed differences are *unlikely* to have occurred by chance.
2. Tests use a **significance level** (typically 5%). Too much emphasis on the "magic" 5% level can be misleading — failure at 5% does not mean the differences are unimportant if they are significant at, say, 10%.

### Watson's Test for Randomness

Watson (1956) demonstrated how to test whether a set of directions is random using the resultant length $R$. Randomly oriented vectors will have a small $R$, while non-random (clustered) directions will have $R$ approaching $N$.

The critical value $R_o$ at the 95% confidence level can be estimated by:

$$R_o = \sqrt{\frac{7.815 \cdot N}{3}}$$

If $R > R_o$, the null hypothesis of randomness can be rejected.

In [ ]:
# Case 1: Clustered directions (should reject randomness)
dirs_clustered = ipmag.fishrot(k=20, n=20, dec=0, inc=50)
mean_clustered = ipmag.fisher_mean(di_block=dirs_clustered)
R_clustered = mean_clustered['r']
Ro = np.sqrt(7.815 * 20 / 3)

plt.figure(figsize=(5, 5))
ipmag.plot_net(fignum=0)
ipmag.plot_di(di_block=dirs_clustered, color='steelblue', markersize=30, edge='k')
ipmag.plot_di_mean(mean_clustered['dec'], mean_clustered['inc'],
                   mean_clustered['alpha95'], color='red', marker='s', markersize=40)
plt.title(f"Clustered (κ = 20, N = 20)\nR = {R_clustered:.2f}, Ro = {Ro:.2f}", fontsize=12)
plt.show()

print(f"R = {R_clustered:.2f}, Ro = {Ro:.2f}")
if R_clustered > Ro:
    print("R > Ro → Reject randomness (directions are significantly clustered)")
else:
    print("R < Ro → Cannot reject randomness")

In [ ]:
# Case 2: Random directions (should not reject randomness)
dirs_random = ipmag.fishrot(k=0.001, n=20, dec=0, inc=50)
mean_random = ipmag.fisher_mean(di_block=dirs_random)
R_random = mean_random['r']
Ro_2 = np.sqrt(7.815 * 20 / 3)

plt.figure(figsize=(5, 5))
ipmag.plot_net(fignum=0)
ipmag.plot_di(di_block=dirs_random, color='steelblue', markersize=30, edge='k')
plt.title(f"Random (κ ≈ 0, N = 20)\nR = {R_random:.2f}, Ro = {Ro_2:.2f}", fontsize=12)
plt.show()

print(f"R = {R_random:.2f}, Ro = {Ro_2:.2f}")
if R_random > Ro_2:
    print("R > Ro → Reject randomness")
else:
    print("R < Ro → Cannot reject randomness (directions are consistent with random)")

### Comparing Two Mean Directions

A common question in paleomagnetism is whether two data sets share a common mean direction. There are two main approaches:

**Watson's F test** (Watson, 1956): The statistic

$$F = (N - 2) \frac{(R_1 + R_2 - R)}{(N - R_1 - R_2)}$$

is compared to the $F$-distribution with 2 and $2(N-2)$ degrees of freedom, where $N = N_1 + N_2$ and $R$ is the resultant of all $N$ directions combined. Intuitively, if two data sets have very different means but are internally well-grouped, $(R_1 + R_2)$ will be much larger than $R$, yielding a large $F$ statistic.

**Watson's $V_w$ test** (Watson, 1983): A more robust statistic whose critical value is determined through Monte Carlo simulation. The McFadden & McElhinny (1990) classification scheme assigns letter grades (A, B, C) based on the angle between the means relative to critical angles.

When confidence circles don't overlap, the means are clearly different. When one circle includes the other mean, they are clearly indistinguishable. The tricky "grey zone" cases — where circles overlap but neither includes the other mean — require the formal tests above.

You will apply these comparison tests using PmagPy functions (`pmag.watsons_f` and `ipmag.common_mean_watson`) in your homework assignment.

## Is a Given Data Set Fisher Distributed?

All of the statistical tests above assume that the data follow a Fisher distribution. But how can we tell? The **quantile-quantile (Q-Q) test** (Fisher et al., 1987) exploits the two properties we verified above:
- Declinations (in data coordinates) should be **uniformly** distributed
- Co-inclinations should be **exponentially** distributed

PmagPy's `ipmag.fishqq()` performs this test, producing Q-Q plots where data compatible with a Fisher distribution fall along a straight line. It also calculates test statistics $M_u$ (for uniform declinations) and $M_e$ (for exponential co-inclinations) that are compared against critical values.

Let's test a data set we *know* is Fisher distributed:

In [ ]:
# Q-Q test on Fisher-distributed data (should pass)
dirs_fisher = ipmag.fishrot(k=20, n=50, dec=0, inc=45)
print("Fisher-distributed data (κ=20, N=50):")
result_fisher = ipmag.fishqq(di_block=dirs_fisher)

Now let's test a data set that is *not* Fisher distributed — a mixture of two clusters, which might arise from inadequate separation of two magnetic components:

In [ ]:
# Q-Q test on mixed clusters (should fail)
dirs_a = ipmag.fishrot(k=50, n=25, dec=0, inc=45)
dirs_b = ipmag.fishrot(k=50, n=25, dec=30, inc=45)
dirs_mixed = dirs_a + dirs_b

print("Mixed two-cluster data (should fail Fisher test):")
result_mixed = ipmag.fishqq(di_block=dirs_mixed)

## Beyond the Fisher Distribution

Fisher statistics assume a **symmetric** distribution of directions about the mean. In practice, many paleomagnetic data sets have an elliptical distribution (wider in one direction than another) rather than the circular symmetry required by Fisher. Several alternatives exist:

- **Kent distribution** (5-parameter Fisher-Bingham distribution): Allows for elliptical confidence regions. Useful when the data are clearly non-circular in distribution.
- **Bootstrap resampling**: A non-parametric approach that makes no assumption about the underlying distribution. PmagPy implements bootstrap tests for common mean (`ipmag.common_mean_bootstrap`) that are applicable even when data are not Fisher distributed.
- **Bingham distribution**: Useful for analyzing data that are distributed along great circles (girdle distributions) rather than clustered around a point.

When the Q-Q test rejects the Fisher distribution, these alternative approaches should be considered.

**Key takeaways:**
1. The Fisher distribution is the spherical analogue of the normal distribution, characterized by the precision parameter $\kappa$
2. The mean direction is calculated by vector addition, not arithmetic averaging of angles
3. $\alpha_{95}$ quantifies uncertainty of the mean estimate and improves with more data; CSD and $k$ estimate intrinsic population properties
4. When data are not Fisher distributed, non-parametric methods (bootstrap) provide robust alternatives (more to come!)

### References

- Fisher, R.A. (1953). Dispersion on a sphere. *Proc. R. Soc. Lond. A*, 217, 295-305.
- Watson, G.S. (1956). A test for randomness of directions. *Monthly Notices of the Royal Astronomical Society, Geophysical Supplement*, 7, 160-161.
- Watson, G.S. (1983). *Statistics on Spheres*. John Wiley & Sons.
- McFadden, P.L. and McElhinny, M.W. (1990). Classification of the reversal test in palaeomagnetism. *Geophys. J. Int.*, 103, 725-729.
- Fisher, N.I., Lewis, T. and Embleton, B.J.J. (1987). *Statistical Analysis of Spherical Data*. Cambridge University Press.
- Tauxe, L. (2010). *Essentials of Paleomagnetism*. University of California Press.